# 05 Dimensionality Reduction (DuckDB Input + BLOB Artifacts)

This notebook creates compact text embeddings from session text and stores reducers plus outputs in DuckDB.

In [1]:
from pathlib import Path
import json
import pickle

import pandas as pd
import plotly.express as px
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer

from utils.duckdb_utils import init_ml_artifacts_table, open_duckdb, query_df
from utils.paths import get_db_paths, resolve_workspace
            


In [2]:
workspace = resolve_workspace(Path.cwd())
input_db, output_db = get_db_paths(workspace)

init_ml_artifacts_table(output_db)
            


IOException: IO Error: Cannot open file "\\?\C:\projects\fab\Fabcon2026\src\Processed\sessions_in_preprocessed.duckdb": The system cannot find the path specified.


In [ ]:
df = query_df(
    input_db,
    '''
    SELECT file, title, day, track, conference, COALESCE(NULLIF(text, ''), description) AS text
    FROM sessions_in_preprocessed
    WHERE COALESCE(NULLIF(text, ''), description) IS NOT NULL
    '''
)

vectorizer = TfidfVectorizer(max_features=5000, min_df=2, ngram_range=(1, 2))
X = vectorizer.fit_transform(df['text'])

n_components = 8
svd = TruncatedSVD(n_components=n_components, random_state=42)
emb = svd.fit_transform(X)

embedding_cols = [f'svd_{i+1}' for i in range(n_components)]
embedding_df = pd.concat([
    df[['file', 'title', 'day', 'track', 'conference']].reset_index(drop=True),
    pd.DataFrame(emb, columns=embedding_cols)
], axis=1)

with open_duckdb(output_db, read_only=False) as write_con:
    write_con.register('embedding_df', embedding_df)
    write_con.execute('CREATE OR REPLACE TABLE dr_embeddings AS SELECT * FROM embedding_df')

    explained = svd.explained_variance_ratio_.tolist()
    metrics = {
        'n_components': n_components,
        'explained_variance_ratio': [float(v) for v in explained],
        'explained_total': float(sum(explained))
    }
    artifact_blob = pickle.dumps({'vectorizer': vectorizer, 'svd': svd})

    write_con.execute('''
    INSERT OR REPLACE INTO ml_artifacts
    (artifact_id, notebook, artifact_type, model_name, metrics_json, artifact_blob)
    VALUES (?, ?, ?, ?, ?, ?)
    ''', [
        'dr_svd_text',
        '05_dimensionality_reduction',
        'dimensionality_reduction_model',
        'TFIDF + TruncatedSVD',
        json.dumps(metrics),
        artifact_blob
    ])

print(f"Explained variance total: {metrics['explained_total']:.4f}")
embedding_df.head()
            


In [ ]:
fig = px.scatter(
    embedding_df,
    x='svd_1',
    y='svd_2',
    color='track',
    hover_name='title',
    hover_data=['day', 'conference'],
    title='Interactive 2D Projection from TruncatedSVD'
)
fig.show()